# 01 — Weekly BO Runner (GP‑UCB)
Run top-to-bottom each week.

In [1]:
# 🔔 Manual Strategy Override Reminder (Run this before proposing Week N points)


from src.config import RUNS_BASE, WEEK
from src.overrides import write_overrides_template

week_dir = RUNS_BASE / f"week_{WEEK:02d}"
csv_path = week_dir / "overrides.csv"

# Create template if missing
if not csv_path.exists():
    csv_path = write_overrides_template(RUNS_BASE, WEEK, function_ids=[1,2,3,4,5,6,7,8])

print("-----------------------------------------------------------")
print(f"📌 You may adjust strategy here (OPTIONAL): {csv_path}")
print(" - Edit kappa_delta to shift explore(+)/exploit(-)")
print(" - Edit tr_halfwidth for local trust region (e.g., 0.10)")
print(" - Edit n_candidates_multiplier to expand candidate sampling (>1)")
print("-----------------------------------------------------------")
print("If you DO NOT want manual strategy changes, continue. ✅")
print("If you WANT manual fine-tuning, edit the CSV before continuing.")

-----------------------------------------------------------
📌 You may adjust strategy here (OPTIONAL): /Users/mrinmoy/Desktop/Mrinmoy/Imperial-College/Capstone Project - Black-Box-Optimization-Capstone/aiml-capstone-project/runs/week_14/overrides.csv
 - Edit kappa_delta to shift explore(+)/exploit(-)
 - Edit tr_halfwidth for local trust region (e.g., 0.10)
 - Edit n_candidates_multiplier to expand candidate sampling (>1)
-----------------------------------------------------------
If you DO NOT want manual strategy changes, continue. ✅
If you WANT manual fine-tuning, edit the CSV before continuing.


In [2]:
from src.config import WEEK, SEED, INITIAL_DATA_ROOT, RUNS_BASE
from src.data_io import list_functions, load_xy, write_csv_with_precision, apply_week_csv
from src.strategy import run_strategy_scan
from src.proposer import propose_points_for_week
from src.visualize import plot_function_snapshot
from src.reflect import generate_reflection_pre_report, generate_reflection_post_results
from src.report import generate_weekly_md_report
import pandas as pd
from datetime import datetime
print('WEEK =', WEEK, '| SEED =', SEED)
week_dir = RUNS_BASE / f"week_{WEEK:02d}"
proposals_path = week_dir / "proposals.csv"


if proposals_path.exists() :
    raise RuntimeError(f"Week {WEEK} has already been run. Found: {proposals_path.resolve()}")

WEEK = 14 | SEED = 123


## 1) Audit current dataset

In [3]:
rows = []
for fdir in list_functions(INITIAL_DATA_ROOT):
    fid = int(fdir.name.split('_')[1])
    X, y = load_xy(fdir)
    rows.append({'function_id': fid, 'n': len(y), 'dim': X.shape[1], 'y_min': float(y.min()), 'y_max': float(y.max())})
audit_df = pd.DataFrame(rows).sort_values('function_id').reset_index(drop=True)
audit_df

,function_id,n,dim,y_min,y_max
0,1,23,2,-0.007215,7.422725e-15
1,2,23,2,-0.065624,6.707739e-01
2,3,28,3,-0.418332,-5.311456e-03
3,4,43,4,-32.625660,6.693065e-01
4,5,33,4,0.112940,4.166259e+03
5,6,33,5,-2.571170,-3.460331e-01
6,7,43,6,0.002701,1.471711e+00
7,8,53,8,5.592193,9.965337e+00


## 2) (Optional) Apply last week's results when new y arrive

In [ ]:
# Example:
#apply_week_csv(RUNS_BASE / f"week_{WEEK-1:02d}", INITIAL_DATA_ROOT)
#from src.reflect import generate_reflection_post_results
#generate_reflection_post_results(WEEK-1)


## 3) Strategy scan (writes overrides JSON)

In [ ]:
_ = run_strategy_scan(WEEK)
(RUNS_BASE / f"week_{WEEK:02d}").mkdir(parents=True, exist_ok=True)
print('Wrote overrides to:', RUNS_BASE / f"week_{WEEK:02d}" / 'strategy_overrides.json')

## 4) Propose next x (one per function)

In [6]:
from datetime import timezone

proposals = propose_points_for_week(SEED, WEEK)
week_dir = RUNS_BASE / f"week_{WEEK:02d}"
week_dir.mkdir(parents=True, exist_ok=True)

df = proposals.copy()
df['y'] = ''
df['status'] = 'pending'
df['method'] = f"gp_ucb_adaptive_wk{WEEK}"
df['created_at'] = datetime.now(timezone.utc).isoformat()
df['applied_at'] = ''

csv_path = week_dir / 'proposals.csv'
write_csv_with_precision(df, csv_path, decimals=6)
print('Saved:', csv_path)
proposals



=== Strategy Selection Report ===
function_1: USING MANUAL override → kappa_delta=-0.6, n_candidates_multiplier=1.0, tr_halfwidth_override=0.06
function_2: USING MANUAL override → kappa_delta=-0.2, n_candidates_multiplier=1.0, tr_halfwidth_override=0.12
function_3: USING MANUAL override → kappa_delta=-0.1, n_candidates_multiplier=1.2, tr_halfwidth_override=0.12
function_4: USING MANUAL override → kappa_delta=0.2, n_candidates_multiplier=1.3, tr_halfwidth_override=0.15
function_5: USING MANUAL override → kappa_delta=0.3, n_candidates_multiplier=1.5, tr_halfwidth_override=0.18
function_6: USING MANUAL override → kappa_delta=0.1, n_candidates_multiplier=1.2, tr_halfwidth_override=0.15
function_7: USING MANUAL override → kappa_delta=-0.2, n_candidates_multiplier=1.0, tr_halfwidth_override=0.12
function_8: USING MANUAL override → kappa_delta=-0.3, n_candidates_multiplier=1.0, tr_halfwidth_override=0.1

base kappa for dimension: 2 1.8
base kappa for dimension: 2 1.8
base kappa for dimension

,function_id,x1,x2,x3,x4,x5,x6,x7,x8,x_concat
0,1,0.023515,0.240559,NaN,NaN,NaN,NaN,NaN,NaN,0.023515-0.240559
1,2,0.696155,0.718272,NaN,NaN,NaN,NaN,NaN,NaN,0.696155-0.718272
2,3,0.740375,0.970277,0.488933,NaN,NaN,NaN,NaN,NaN,0.740375-0.970277-0.488933
3,4,0.444245,0.347714,0.412230,0.421340,NaN,NaN,NaN,NaN,0.444245-0.347714-0.412230-0.421340
4,5,0.836860,0.878855,0.946850,0.990765,NaN,NaN,NaN,NaN,0.836860-0.878855-0.946850-0.990765
5,6,0.198452,0.336288,0.789924,0.908998,0.204299,NaN,NaN,NaN,0.198452-0.336288-0.789924-0.908998-0.204299
6,7,0.652530,0.911751,0.887736,0.097316,0.801499,0.732558,NaN,NaN,0.652530-0.911751-0.887736-0.097316-0.801499-0...
7,8,0.225186,0.061911,0.089349,0.148527,0.960079,0.904460,0.153725,0.212632,0.225186-0.061911-0.089349-0.148527-0.960079-0...


## 5) Visuals

In [ ]:
plots_dir = (RUNS_BASE / f"week_{WEEK:02d}" / 'plots')
plots_dir.mkdir(parents=True, exist_ok=True)

for fdir in list_functions(INITIAL_DATA_ROOT):
    fid = int(fdir.name.split('_')[1])
    try:
        # Generate main snapshot plot
        plot_function_snapshot(fid, save_dir=plots_dir)
        print(f"✓ Generated snapshot for function {fid}")

        # Generate PCA plot for higher dimensions
        X, _ = load_xy(fdir)
        if X.shape[1] >= 3:
            from src.visualize import plot_pca_scatter
            plot_pca_scatter(fid, save_dir=plots_dir)
            print(f"✓ Generated PCA plot for function {fid}")
    except Exception as e:
        print(f"✗ Plot failed for function {fid}: {e}")

print(f"\n📊 Plots saved to: {plots_dir}")
print(f"📁 Files generated: {list(plots_dir.glob('*.png'))}")


## 6) Reflections & Weekly Report

In [ ]:
_ = generate_reflection_pre_report(WEEK)
report_path = generate_weekly_md_report(WEEK)
print('Report:', report_path)


## 7) 🎉 Week {WEEK} proposals are ready!
Submit `runs/week_{WEEK:02d}/proposals.csv` to the evaluation portal, then wait for results.


---
# ⬇️ RUN THE CELLS BELOW AFTER WEEK {WEEK} RESULTS ARRIVE ⬇️
---


## 8) Apply Week {WEEK} Results (After receiving y values)

In [4]:
# STEP 1: Fill the 'y' column in runs/week_{WEEK:02d}/proposals.csv with the received values
# STEP 2: Then run this cell to apply results to initial_data

from src.analysis import check_week_performance, analyze_trends_for_next_week, compare_override_strategies

apply_week_csv(RUNS_BASE / f"week_{WEEK:02d}", INITIAL_DATA_ROOT)
generate_reflection_post_results(WEEK)

print("\n" + "="*80)
print("✅ Week {WEEK} results applied to initial_data/")
print("="*80)



✅ Week {WEEK} results applied to initial_data/


## 9) Performance Analysis & Next Week Planning

In [5]:
# Analyze Week {WEEK} performance
print("\n📊 WEEK {WEEK} PERFORMANCE CHECK")
print("="*80)
check_week_performance(WEEK)



📊 WEEK {WEEK} PERFORMANCE CHECK

WEEK 13 PERFORMANCE CHECK

Function     Week 12         Week 13         Change       Status    
--------------------------------------------------------------------------------
Function 1    7.422725e-15    -9.406547e-134   -100.0%    ⚠️ DECLINED
Function 2    6.126310e-01    6.139761e-01       +0.2%    ⏸️ STABLE
Function 3    -2.216116e-02   -3.700552e-02     -67.0%    ⚠️ DECLINED
Function 4    6.010982e-01    3.199952e-01      -46.8%    ⚠️ DECLINED
Function 5    3.874894e+03    3.992581e+03       +3.0%    ⏸️ STABLE
Function 6    -3.460331e-01   -6.742274e-01     -94.8%    ⚠️ DECLINED
Function 7    2.792105e-01    3.834495e-03      -98.6%    ⚠️ DECLINED
Function 8    9.829980e+00    9.760249e+00       -0.7%    ⏸️ STABLE



In [6]:
# Compare Week {WEEK-1} vs Week {WEEK} strategies
print("\n📊 STRATEGY COMPARISON")
print("="*80)
compare_override_strategies(WEEK-1, WEEK)



📊 STRATEGY COMPARISON

OVERRIDE COMPARISON: Week 12 vs Week 13

Fn   W12 κ    W13 κ    Δκ       W12 TR   W13 TR   ΔTR      Strategy Shift                
--------------------------------------------------------------------------------
F1    -0.500   -0.600   -0.100    0.080    0.060   -0.020  MAINTAIN APPROACH →           
F2     0.300   -0.200   -0.500    0.180    0.120   -0.060  MORE EXPLOITATION ↓           
F3     0.200   -0.100   -0.300    0.150    0.120   -0.030  MORE EXPLOITATION ↓           
F4    -0.300    0.200   +0.500    0.100    0.150   +0.050  MORE EXPLORATION ↑            
F5    -0.400    0.300   +0.700    0.090    0.180   +0.090  MORE EXPLORATION ↑            
F6     0.500    0.100   -0.400    0.250    0.150   -0.100  MORE EXPLOITATION ↓           
F7     0.700   -0.200   -0.900    0.280    0.120   -0.160  MORE EXPLOITATION ↓           
F8    -0.200   -0.300   -0.100    0.120    0.100   -0.020  MAINTAIN APPROACH →           



In [7]:
# Analyze trends to prepare for Week {WEEK+1}
print("\n📊 PREPARING FOR WEEK {WEEK+1}")
print("="*80)
analyze_trends_for_next_week(WEEK)

print("\n" + "="*80)
print(f"💡 NEXT STEPS:")
print(f"   1. Review the analysis above")
print(f"   2. Set WEEK = {WEEK+1} in src/config.py")
print(f"   3. Adjust runs/week_{WEEK+1:02d}/overrides.csv if needed")
print(f"   4. Re-run cells 1→6 to generate Week {WEEK+1} proposals")
print("="*80)



📊 PREPARING FOR WEEK {WEEK+1}
ANALYZING TRENDS FOR WEEK 14 RECOMMENDATIONS

Recent Performance Trends:
--------------------------------------------------------------------------------
Fn   W{current_week-1} W{current_week} Change       Status              
--------------------------------------------------------------------------------
F1   7.422725e-15    -9.406547e-134   -100.0%    ⚠️ BIG DROP         
F2   6.126310e-01    6.139761e-01       +0.2%    ⏸️ STABLE           
F3   -2.216116e-02   -3.700552e-02     -67.0%    ⚠️ BIG DROP         
F4   6.010982e-01    3.199952e-01      -46.8%    ⚠️ BIG DROP         
F5   3.874894e+03    3.992581e+03       +3.0%    ⏸️ STABLE           
F6   -3.460331e-01   -6.742274e-01     -94.8%    ⚠️ BIG DROP         
F7   2.792105e-01    3.834495e-03      -98.6%    ⚠️ BIG DROP         
F8   9.829980e+00    9.760249e+00       -0.7%    ⏸️ STABLE           

💡 Next steps:
   1. Review trends above
   2. Manually edit runs/week_14/overrides.csv based on patt